# DiMEx: exploratory data analysis of the SemEval-2026 Task 3 (DimABSA) data

The English Track A data that DiMEx's first training stage learns valence and arousal from: restaurant and laptop
review sentences in which each aspect is scored for **valence** (1 = very negative, 9 = very positive) and
**arousal** (1 = calm, 9 = intense).

No GPU is needed (*Runtime → Change runtime type → CPU* is fine). Run all cells in order; each section ends with a
short "what to look for" note. Figures are saved to `My Drive/dimex/eda/semeval/` for the report and slides.

| Section | Question |
|---|---|
| 1–2 | Setup; loading every English file, train, dev and test |
| 3 | How big is it, and where do the sentences come from? |
| 4 | How many aspects are named in the text, and how many are implied? |
| 5 | How are valence and arousal distributed? |
| 6 | How do valence and arousal relate to each other? |
| 7 | Do restaurant and laptop reviews differ? |
| 8 | Which categories and aspects dominate? |
| 9 | How long are the sentences, and does the 256-token limit cut anything? |
| 10 | Is the training data like the dev and test data? |
| 11 | Where do DiMEx's Druglib anchor values land on the SemEval map? |
| 12 | Summary table to export |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = '/content/DiMEx'
WORK = '/content/drive/MyDrive/dimex'
FIGS = f'{WORK}/eda/semeval'
os.makedirs(FIGS, exist_ok=True)

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/av-jones/DiMEx.git', REPO], check=True)
os.chdir(REPO)

if not os.path.exists('/content/DimABSA2026'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/DimABSA/DimABSA2026.git', '/content/DimABSA2026'], check=True)
ENG = '/content/DimABSA2026/task-dataset/track_a/subtask_1/eng'
print(sorted(os.listdir(ENG)))

In [ ]:
import glob, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
C = {'restaurant': '#2a78d6', 'laptop': '#d4537e', 'train': '#5f5e5a', 'dev': '#1d9e75', 'test': '#eb6834',
     'explicit': '#2a78d6', 'implicit': '#b4b2a9'}

def save(fig, name):
    fig.savefig(f'{FIGS}/{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

## 2. Load every English file

Each line of a file is one sentence. Training files (`*_train_alltasks.jsonl`) list each aspect as a
*quadruplet*: the aspect term, its category, the opinion words and the valence-arousal score. Dev and test files
(`*_task1.jsonl`) list only the aspect and its score. Two tables come out of this:

- `sents`: one row per sentence
- `asp`: one row per aspect, the unit the model is trained on (a sentence with three aspects gives three training examples)

`ID` prefixes record where each sentence came from (`rest16_quad_train_…`, `lap26_aspect_va_test_…`).

In [ ]:
def parse_file(path):
    name = os.path.basename(path)
    domain = 'restaurant' if 'restaurant' in name else 'laptop'
    split = 'train' if 'train' in name else 'dev' if '_dev_' in name else 'test'
    s_rows, a_rows = [], []
    for line in open(path, encoding='utf-8'):
        if not line.strip():
            continue
        r = json.loads(line)
        items = r.get('Quadruplet') or r.get('Triplet') or r.get('Aspect_VA') or []
        s_rows.append({'id': r['ID'], 'domain': domain, 'split': split, 'text': r['Text'],
                       'source': r['ID'].rsplit('_', 1)[0], 'n_aspects': len(items)})
        for it in items:
            if 'VA' not in it:
                continue
            v, a = map(float, it['VA'].split('#'))
            aspect = it.get('Aspect')
            a_rows.append({'id': r['ID'], 'domain': domain, 'split': split,
                           'aspect': aspect, 'implicit': aspect in (None, 'NULL'),
                           'category': it.get('Category'), 'opinion': it.get('Opinion'),
                           'opinion_implicit': it.get('Opinion') == 'NULL',
                           'v': v, 'a': a})
    return s_rows, a_rows

S, A = [], []
for path in sorted(glob.glob(f'{ENG}/*.jsonl')):
    s, a = parse_file(path)
    S += s; A += a
sents, asp = pd.DataFrame(S), pd.DataFrame(A)
sents['n_words'] = sents.text.str.split().str.len()
asp = asp.merge(sents[['id', 'split', 'text', 'n_words']], on=['id', 'split'])

# The model sees an implied aspect as its category in words ("RESTAURANT#GENERAL" -> "restaurant general")
asp['aspect_text'] = np.where(asp.implicit,
                              asp.category.fillna('').str.replace('#', ' ').str.replace('_', ' ').str.lower(),
                              asp.aspect)
# The training pipeline drops exact duplicates (same sentence, aspect text and score); count them the same way
asp['dup'] = asp.duplicated(['split', 'id', 'text', 'aspect_text', 'v', 'a'])
train = asp[(asp.split == 'train') & ~asp.dup]
print(f'{len(sents):,} sentences, {len(asp):,} aspects '
      f'({asp.dup.sum()} exact duplicates); training examples after de-duplication: {len(train):,}')
display(asp.head())

## 3. Size and where the sentences come from

In [ ]:
size = (sents.groupby(['split', 'domain'])
        .agg(sentences=('id', 'size'), aspects=('n_aspects', 'sum'),
             aspects_per_sentence=('n_aspects', 'mean'))
        .round(2).reindex(['train', 'dev', 'test'], level=0))
display(size)

print('Sentence sources (from ID prefixes):')
display(sents.groupby(['split', 'domain', 'source']).size().rename('sentences').to_frame())

fig, ax = plt.subplots(figsize=(6, 3.2))
per = sents[sents.split == 'train'].groupby('domain').n_aspects.value_counts().unstack(0).fillna(0)
per.index = [str(i) if i < 6 else '6+' for i in per.index]
per = per.groupby(level=0, sort=False).sum()
per.plot.bar(ax=ax, color=[C['laptop'], C['restaurant']], width=0.8, rot=0)
ax.set_xlabel('Aspects in a sentence'); ax.set_ylabel('Training sentences')
ax.set_title('Aspects per training sentence')
save(fig, '01_aspects_per_sentence')

**What to look for:** the training files merge three older splits (`*_quad_train`, `*_quad_dev`, `*_quad_test`).
The task README says they are the ACOS dataset (Cai et al., 2021) re-annotated with valence and arousal. The dev
and test files (`*26_aspect_va_*`) are new 2026 sentences, so they are a fair held-out check. Most sentences have
one or two aspects, so the model mostly sees short, focused judgments.

## 4. Named vs. implied aspects

In [ ]:
imp = (asp[~asp.dup].groupby(['split', 'domain'])
       .agg(aspects=('implicit', 'size'), implicit=('implicit', 'sum'),
            implicit_share=('implicit', 'mean'), opinion_implicit_share=('opinion_implicit', 'mean'))
       .round(3).reindex(['train', 'dev', 'test'], level=0))
display(imp)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, dim, label in [(axes[0], 'v', 'Valence'), (axes[1], 'a', 'Arousal')]:
    for kind, g in train.groupby('implicit'):
        key = 'implicit' if kind else 'explicit'
        ax.hist(g[dim], bins=np.arange(0.75, 9.5, 0.5), alpha=0.65, density=True, color=C[key],
                label=f'{key} (n={len(g):,})')
    ax.set_xlabel(label); ax.set_xlim(0.8, 9.2)
axes[0].set_ylabel('Share of training aspects'); axes[0].legend(frameon=False)
fig.suptitle('Named (explicit) vs. implied (implicit) aspects, training data')
save(fig, '02_explicit_vs_implicit')

**What to look for:** an *implicit* aspect has `Aspect = NULL`: the sentence judges something it never names
("can't wait for my next visit" → restaurant in general). The DiMEx loader keeps them, replacing NULL with the
category in words ("restaurant general"), because they look most like DiMEx's drug aspects ("Effectiveness",
"Satisfaction"), which are never named word for word in a review. The dev and test files contain no implicit
aspects at all, so the model is never tested on this kind of example.

## 5. Valence and arousal distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
bins = np.arange(0.875, 9.2, 0.25)
for ax, dim, label in [(axes[0], 'v', 'Valence (1 negative → 9 positive)'),
                       (axes[1], 'a', 'Arousal (1 calm → 9 intense)')]:
    ax.hist(train[dim], bins=bins, color=C['train'], alpha=0.85)
    ax.axvline(train[dim].mean(), color='black', lw=1, ls='--')
    ax.set_xlabel(label); ax.set_xlim(0.8, 9.2)
    ax.text(0.02, 0.95, f"mean {train[dim].mean():.2f}\nmedian {train[dim].median():.2f}\nsd {train[dim].std():.2f}",
            transform=ax.transAxes, va='top', fontsize=9)
axes[0].set_ylabel('Training aspects')
fig.suptitle('Valence and arousal, training aspects')
save(fig, '03_va_histograms')

desc = train[['v', 'a']].describe(percentiles=[.05, .25, .5, .75, .95]).T.round(2)
desc.index = ['valence', 'arousal']
display(desc)
for dim, name in [('v', 'valence'), ('a', 'arousal')]:
    print(f"{name}: {(train[dim] < 4).mean():.1%} below 4, {(train[dim].between(4, 6)).mean():.1%} between 4 and 6, "
          f"{(train[dim] > 6).mean():.1%} above 6; exactly 5.00: {(train[dim] == 5).mean():.1%}")

**What to look for:** valence is usually spread out and skewed positive (people review places they like);
arousal is usually squeezed toward the middle. A narrow arousal range limits how much the model can learn about
intensity. Note the share of scores at exactly 5.00, the neutral default: in the sample sentence in section 2,
"service general" is scored 5.00#5.00.

## 6. How valence and arousal relate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
hb = axes[0].hexbin(train.v, train.a, gridsize=32, extent=(1, 9, 1, 9), cmap='Blues', mincnt=1, bins='log')
fig.colorbar(hb, ax=axes[0], label='Aspects (log scale)')
axes[0].set_xlabel('Valence'); axes[0].set_ylabel('Arousal'); axes[0].set_title('All training aspects')
for ax in axes:
    ax.axvline(5, color='#888', lw=0.8, ls=':'); ax.axhline(5, color='#888', lw=0.8, ls=':')
    ax.set_xlim(1, 9); ax.set_ylim(1, 9)

# Arousal as a function of valence: the classic V-shape, if present
bins = pd.cut(train.v, np.arange(1, 9.5, 0.5))
curve = train.groupby(bins, observed=True).a.agg(['mean', 'count'])
mids = [b.mid for b in curve.index]
axes[1].plot(mids, curve['mean'], '-o', color=C['train'], ms=4)
axes[1].set_xlabel('Valence'); axes[1].set_ylabel('Mean arousal'); axes[1].set_title('Mean arousal by valence')
save(fig, '04_va_joint')

r = train[['v', 'a']].corr().iloc[0, 1]
extremity = (train.v - 5).abs()
r2 = np.corrcoef(extremity, train.a)[0, 1]
print(f'Correlation of valence with arousal: r = {r:.2f}')
print(f'Correlation of |valence - 5| (how far from neutral) with arousal: r = {r2:.2f}')

**What to look for:** in most emotion data, arousal rises at *both* ends of valence (strong praise and strong
complaints are both intense), giving a V or U shape, and the correlation with |valence − 5| is much stronger
than with valence itself. If that holds here, much of what the model learns as "arousal" is really "how far from
neutral", which matters for DiMEx's side-effect intensity: it would struggle to tell *mild but very negative*
from *intense and very negative*.

## 7. Restaurant vs. laptop

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, dim, label in [(axes[0], 'v', 'Valence'), (axes[1], 'a', 'Arousal')]:
    data = [train[train.domain == d][dim] for d in ('restaurant', 'laptop')]
    parts = ax.violinplot(data, showmedians=True, widths=0.8)
    for body, d in zip(parts['bodies'], ('restaurant', 'laptop')):
        body.set_facecolor(C[d]); body.set_alpha(0.6)
    ax.set_xticks([1, 2], ['Restaurant', 'Laptop']); ax.set_title(label); ax.set_ylim(1, 9)
fig.suptitle('Valence and arousal by domain, training aspects')
save(fig, '05_by_domain')

by_dom = train.groupby('domain')[['v', 'a']].agg(['mean', 'std']).round(2)
by_dom['n'] = train.groupby('domain').size()
display(by_dom)

**What to look for:** large differences between domains would mean the model learns domain habits along with sentiment. Neither domain is medical, which is the domain gap DiMEx has to cross in stage 2.

## 8. Categories and aspects

In [ ]:
cat = (train.dropna(subset=['category']).groupby(['domain', 'category'])
       .agg(n=('v', 'size'), valence=('v', 'mean'), arousal=('a', 'mean')).round(2)
       .sort_values('n', ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, d in zip(axes, ('restaurant', 'laptop')):
    top = cat.loc[d].head(12).iloc[::-1]
    ax.barh(top.index, top.n, color=C[d])
    ax.set_title(f'{d.title()}: 12 most common categories'); ax.set_xlabel('Training aspects')
    ax.tick_params(axis='y', labelsize=8)
save(fig, '06_top_categories')
display(cat.groupby(level=0).head(8))

print('Most frequent named aspects:')
named = train[~train.implicit]
display(named.groupby('domain').aspect.apply(lambda s: s.str.lower().value_counts().head(10)).rename('count').to_frame())

**What to look for:** a few categories (food quality, service, laptop general) usually dominate. The model sees many examples of judging *quality* and *general* satisfaction, and relatively few about anything like physical side effects.

## 9. Sentence length and the 256-token limit

DiMEx feeds the model the pair *(sentence, aspect)* and truncates at 256 tokens, measured with the RoBERTa
tokenizer (the subword counts differ slightly for other models).

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('roberta-base')
pairs = train.copy()
lens = [len(tok(t, a)['input_ids']) for t, a in zip(pairs.text, pairs.aspect_text)]
pairs['tokens'] = lens

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(pairs.tokens, bins=40, color=C['train'])
ax.axvline(256, color=C['test'], lw=1.5, label='256-token limit')
ax.set_xlabel('Tokens per (sentence, aspect) pair'); ax.set_ylabel('Training examples'); ax.legend(frameon=False)
ax.set_title('Input length')
save(fig, '07_token_lengths')
print(pairs.tokens.describe(percentiles=[.5, .95, .99]).round(1).to_string())
print(f'Examples over 256 tokens: {(pairs.tokens > 256).sum()} of {len(pairs):,}')

**What to look for:** SemEval sentences are short, so truncation should cut nothing. Drug reviews are much longer; compare with Jo's numbers for Druglib and Drugs.com.

## 10. Training vs. dev and test

The dev and test files carry valence-arousal labels too. If their distributions match training, the 10%
validation slice DiMEx cut from the training files was a fair stand-in; they could also serve as an extra
held-out SemEval check for stage 1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
bins = np.arange(0.875, 9.2, 0.25)
for ax, dim, label in [(axes[0], 'v', 'Valence'), (axes[1], 'a', 'Arousal')]:
    for split in ('train', 'dev', 'test'):
        g = asp[(asp.split == split) & ~asp.dup]
        ax.hist(g[dim], bins=bins, density=True, histtype='step', lw=1.6, color=C[split], label=f'{split} (n={len(g):,})')
    ax.set_xlabel(label); ax.set_xlim(0.8, 9.2)
axes[0].set_ylabel('Share of aspects'); axes[0].legend(frameon=False)
fig.suptitle('Training vs. dev and test distributions')
save(fig, '08_train_vs_heldout')

cmp = (asp[~asp.dup].groupby('split')
       .agg(aspects=('v', 'size'), valence_mean=('v', 'mean'), valence_sd=('v', 'std'),
            arousal_mean=('a', 'mean'), arousal_sd=('a', 'std'), implicit_share=('implicit', 'mean'),
            words_per_sentence=('n_words', 'mean'))
       .reindex(['train', 'dev', 'test']).round(2))
display(cmp)

## 11. The bridge to drug reviews: where DiMEx's anchors land

In stage 2, each Druglib rating is converted to a fixed valence-arousal point (`core/va_anchors.py`). Side
effects get both valence and arousal; effectiveness and satisfaction get valence only. This plot places those
anchors on the SemEval training data. Anchors that sit where SemEval has plenty of data ask the model for
something it has seen; anchors in empty regions ask for combinations it never learned.

In [ ]:
from core.va_anchors import EFFECT_VA, SIDE_VA, satisfaction_va
side_names = {1: 'none', 2: 'mild', 3: 'moderate', 4: 'severe', 5: 'extremely severe'}

fig, ax = plt.subplots(figsize=(7.2, 6))
hb = ax.hexbin(train.v, train.a, gridsize=32, extent=(1, 9, 1, 9), cmap='Greys', mincnt=1, bins='log', alpha=0.8)
xs = [SIDE_VA[k][0] for k in sorted(SIDE_VA)]; ys = [SIDE_VA[k][1] for k in sorted(SIDE_VA)]
ax.plot(xs, ys, '-', color=C['test'], lw=1.5, zorder=3)
ax.scatter(xs, ys, s=80, color=C['test'], edgecolor='white', zorder=4, label='Side-effect levels (V and A)')
for k, x, y in zip(sorted(SIDE_VA), xs, ys):
    ax.annotate(side_names[k], (x, y), xytext=(8, 4), textcoords='offset points', fontsize=9, color=C['test'])
for k, (v, _) in EFFECT_VA.items():
    ax.axvline(v, ymax=0.06, color=C['explicit'], lw=3)
ax.plot([], [], color=C['explicit'], lw=3, label='Effectiveness levels (valence only)')
ax.set_xlim(1, 9); ax.set_ylim(1, 9); ax.set_xlabel('Valence'); ax.set_ylabel('Arousal')
ax.set_title('Druglib anchors on the SemEval training data')
ax.legend(frameon=False, loc='upper right', fontsize=9)
save(fig, '09_anchors_on_semeval')

# How much SemEval data sits near each side-effect anchor (within 1 point on both scales)?
rows = []
for k, (v, a) in sorted(SIDE_VA.items()):
    near = train[(train.v.sub(v).abs() <= 1) & (train.a.sub(a).abs() <= 1)]
    rows.append({'side effects': side_names[k], 'valence': v, 'arousal': a,
                 'SemEval aspects within ±1': len(near), 'share of training': round(len(near) / len(train), 3)})
display(pd.DataFrame(rows))

**What to look for:** compare the counts in this table. If the *severe* and *extremely severe* anchors (low
valence, high arousal) have few SemEval examples nearby, stage 1 taught the model little about that corner, which
is one candidate explanation for why the SemEval step slightly hurt Druglib side-effect scores. Effectiveness and
satisfaction anchors only constrain valence, so check whether SemEval's valence range covers all of them.

## 12. Summary table for the report

In [ ]:
summary = pd.DataFrame({
    'domain': ['restaurant', 'laptop', 'all'],
    'train sentences': [*(sents[(sents.split == 'train') & (sents.domain == d)].shape[0] for d in ('restaurant', 'laptop')),
                        sents[sents.split == 'train'].shape[0]],
    'train aspects': [*(train[train.domain == d].shape[0] for d in ('restaurant', 'laptop')), len(train)],
    'implicit share': [*(round(train[train.domain == d].implicit.mean(), 3) for d in ('restaurant', 'laptop')),
                       round(train.implicit.mean(), 3)],
    'valence mean (sd)': [*(f"{train[train.domain == d].v.mean():.2f} ({train[train.domain == d].v.std():.2f})"
                            for d in ('restaurant', 'laptop')), f'{train.v.mean():.2f} ({train.v.std():.2f})'],
    'arousal mean (sd)': [*(f"{train[train.domain == d].a.mean():.2f} ({train[train.domain == d].a.std():.2f})"
                            for d in ('restaurant', 'laptop')), f'{train.a.mean():.2f} ({train.a.std():.2f})'],
    'dev / test sentences': [*(f"{sents[(sents.split == 'dev') & (sents.domain == d)].shape[0]} / "
                               f"{sents[(sents.split == 'test') & (sents.domain == d)].shape[0]}"
                               for d in ('restaurant', 'laptop')),
                             f"{(sents.split == 'dev').sum()} / {(sents.split == 'test').sum()}"],
})
display(summary)
summary.to_csv(f'{FIGS}/semeval_summary.csv', index=False)
print('Figures and summary saved to', FIGS)
print(summary.to_latex(index=False, float_format='%.3f'))   # paste into Overleaf (needs \usepackage{booktabs})